# Pump Scenario Sample Generator & Visualiser

Runs all **10 pump scenarios** (1 normal + 3 failures + 6 decoys) serially using NASA's **progpy CentrifugalPump** model with:
- Cycling voltage load (1h period) for natural signal variation
- Per-signal `process_noise` and `measurement_noise` for realistic sensor jitter
- Wear rates set via `x0` initial state (`wA=0.01` → failure ~6h)

| Group | Failure mode | Discriminating signals | Hard negatives |
|-------|-------------|----------------------|----------------|
| Bearing | `wA=0.01` | shaft speed ↓ **AND** temperature ↑ | setpoint step/ramp (speed ↓, temp flat) |
| Impeller | `wA=0.01` | flow_out ↓ **AND** Qin/Qout ratio > 1 | back-pressure step/ramp (ratio ≈ 1) |
| Seal | `wA=0.01` | flow_out ↓ **AND** flow_in stable (gap grows) | flow-reduction step/ramp (gap ≈ 0) |

In [ ]:
duration_days = 365
save_freq_s = 60
decoy_freq_per_day = 2.0
maintenance_freq_per_month = 1.0
seed = 42
run_short_scenarios = True
run_long_series = True
downsample = 50
failure_severity = 0
ambient_var_K = 5.0
duty_cycle_var = 0.5

In [ ]:
import sys, importlib, warnings
warnings.filterwarnings('ignore')

# Make sample_generator importable and reload if already loaded
if "sample_generator" in sys.modules:
    importlib.reload(sys.modules["sample_generator"])
from sample_generator import build_scenarios, run_all

## Run all 9 simulations
Each scenario runs ~360 steps (6 h at 1 sample/min). Faulty scenarios stop early at the failure threshold.

In [ ]:
import os, glob

if run_short_scenarios:
    # Check if short scenario CSVs already exist
    existing = glob.glob("sample_data/normal.csv")
    if existing and os.path.exists("sample_data/scenarios.csv"):
        print("Loading existing short scenario data from sample_data/*.csv")
        import pandas as pd
        scenarios_meta = pd.read_csv("sample_data/scenarios.csv")
        results = {}
        for _, row in scenarios_meta.iterrows():
            path = f"sample_data/{row['scenario']}.csv"
            if os.path.exists(path):
                results[row["scenario"]] = pd.read_csv(path)
        print(f"  Loaded {len(results)} scenarios")
    else:
        print("Generating short scenarios...")
        scenarios = build_scenarios()
        results = run_all(scenarios)
else:
    results = None
    print("Skipped short scenarios (run_short_scenarios=False)")

In [ ]:
if results:
    import pandas as pd
    rows = []
    for name, df in results.items():
        vc = df["label"].value_counts()
        rows.append({
            "scenario": name, "rows": len(df),
            "NORMAL": vc.get("NORMAL", 0),
            "PRE_FAILURE": vc.get("PRE_FAILURE", 0),
            "FAILURE": vc.get("FAILURE", 0),
        })
    display(pd.DataFrame(rows).set_index("scenario"))

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Image
from sample_generator import COLOURS, LABEL_ALPHA, LABEL_COLOUR

def show_fig(fig):
    """Show interactive plotly + static PNG fallback (for GitHub rendering)."""
    fig.show()
    display(Image(fig.to_image(format="png", width=1400, height=500, scale=2)))

## Plot 0 — Healthy baseline
All signals stable, every row labelled NORMAL. This is what "nothing is wrong" looks like.

In [ ]:
if results:
    df = results["normal"]
    pairs = [
        ("shaft_speed_rads", "Shaft speed (rad/s)"),
        ("fluid_temp_K",     "Temperature (K)"),
        ("flow_out_m3s",     "Flow out (m\u00b3/s)"),
        ("flow_in_m3s",      "Flow in (m\u00b3/s)"),
        ("pump_speed_rads",  "Pump speed (rad/s)"),
        ("thrust_bearing_K", "Thrust bearing Tt (K)"),
    ]
    fig = make_subplots(rows=2, cols=3, subplot_titles=[p[1] for p in pairs])
    for idx, (col, label) in enumerate(pairs):
        r, c = idx // 3 + 1, idx % 3 + 1
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df["time_h"], y=df[col], mode="lines",
                                     line=dict(color=COLOURS["normal"], width=1.5),
                                     name=label, showlegend=False), row=r, col=c)
            fig.update_xaxes(title_text="Time (hours)", row=r, col=c)
    fig.update_layout(title="Healthy pump \u2014 baseline signals (all NORMAL)",
                      height=500, template="plotly_white")
    show_fig(fig)

## Plot 1 — Thrust bearing wear vs high-load decoys

**Both** show Tt rising — but the failure rises **faster** than the operating conditions explain.

- **Failure:** Tt climbs with no speed increase — excess heat from friction (`rThrust * w^2`)
- **Hard negative:** Higher voltage → speed ↑ → Tt rises too, but proportionally — no excess heat

The discrimination problem: you can't just threshold on "Tt is rising" — you need to check whether the rate of rise is explained by the current load.

In [ ]:
def plot_group_interactive(scenarios, signal_y, signal_x, title, y_label, x_label, scatter_label):
    """3-panel interactive plot: signal_y over time, signal_x over time, scatter x vs y."""
    fig = make_subplots(rows=1, cols=3,
                        subplot_titles=[f"{y_label} over time",
                                        f"{x_label} over time",
                                        f"{scatter_label}"])
    for name, label, colour in scenarios:
        df = results[name]
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[signal_y], mode="lines",
                                 line=dict(color=colour, width=1.5),
                                 name=label, legendgroup=label), row=1, col=1)
        fig.add_trace(go.Scatter(x=df["time_h"], y=df[signal_x], mode="lines",
                                 line=dict(color=colour, width=1.5),
                                 name=label, legendgroup=label, showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=df[signal_x], y=df[signal_y], mode="markers",
                                 marker=dict(color=colour, size=3, opacity=0.5),
                                 name=label, legendgroup=label, showlegend=False), row=1, col=3)
    fig.update_xaxes(title_text="Time (hours)", row=1, col=1)
    fig.update_xaxes(title_text="Time (hours)", row=1, col=2)
    fig.update_xaxes(title_text=x_label, row=1, col=3)
    fig.update_yaxes(title_text=y_label, row=1, col=1)
    fig.update_yaxes(title_text=x_label, row=1, col=2)
    fig.update_yaxes(title_text=y_label, row=1, col=3)
    fig.update_layout(title=title, height=450, template="plotly_white")
    return fig

if results:
    fig = plot_group_interactive(
        scenarios=[
            ("normal",              "Healthy",        COLOURS["normal"]),
            ("pump_bearing_wear",   "Bearing wear",   COLOURS["failure"]),
            ("decoy_highload_step", "High-load step", COLOURS["decoy"]),
            ("decoy_highload_ramp", "High-load ramp", "#2196F3"),
        ],
        signal_y="thrust_bearing_K", signal_x="shaft_speed_rads",
        title="Thrust bearing wear vs high-load decoys<br><sub>Failure: Tt higher than expected for its speed | Decoy: Tt tracks speed proportionally</sub>",
        y_label="Thrust bearing Tt (K)", x_label="Shaft speed (rad/s)",
        scatter_label="Speed vs Tt (discriminator)")
    show_fig(fig)

## Plot 2 — Impeller wear vs back-pressure decoys (observable signals only)

**Both** show flow dropping — but only the failure shows temperature diverging from the healthy baseline at the same shaft speed.

- **Failure:** flow drops + temp rises → hidden `A` degradation (not directly measurable)
- **Hard negative:** flow drops because back-pressure changed — temp stays near healthy

You can't just watch flow. You need to check: "is the temperature rise explained by the operating conditions, or is there unexplained excess heat?"

In [ ]:
if results:
    fig = plot_group_interactive(
        scenarios=[
            ("normal",                   "Healthy",         COLOURS["normal"]),
            ("pump_impeller_wear",       "Impeller wear",   COLOURS["failure"]),
            ("decoy_back_pressure_step", "Back-press step", COLOURS["decoy"]),
            ("decoy_back_pressure_ramp", "Back-press ramp", "#2196F3"),
        ],
        signal_y="fluid_temp_K", signal_x="flow_out_m3s",
        title="Impeller wear vs back-pressure decoys<br><sub>Failure: temp higher than expected for its flow | Decoy: temp tracks flow proportionally</sub>",
        y_label="Oil temperature To (K)", x_label="Flow out (m\u00b3/s)",
        scatter_label="Flow vs To (discriminator)")
    show_fig(fig)

## Plot 3 — Radial bearing wear vs high-load decoys

Same logic as thrust bearing: **both** show Tr rising, but the failure has excess heat not explained by load.

- **Failure:** Tr climbs with no speed increase — friction from degraded radial bearing
- **Hard negative:** Higher load → speed ↑ → Tr rises proportionally

In [ ]:
if results:
    fig = plot_group_interactive(
        scenarios=[
            ("normal",                       "Healthy",        COLOURS["normal"]),
            ("pump_radial_wear",             "Radial wear",    COLOURS["failure"]),
            ("decoy_radial_highload_step",   "High-load step", COLOURS["decoy"]),
            ("decoy_radial_highload_ramp",   "High-load ramp", "#2196F3"),
        ],
        signal_y="radial_bearing_K", signal_x="shaft_speed_rads",
        title="Radial bearing wear vs high-load decoys<br><sub>Failure: Tr higher than expected for its speed | Decoy: Tr tracks speed proportionally</sub>",
        y_label="Radial bearing Tr (K)", x_label="Shaft speed (rad/s)",
        scatter_label="Speed vs Tr (discriminator)")
    show_fig(fig)

---
# Long Time Series (365 days)

Generate 1-year data with decoy events mixed into every series (~2/day). Each series saved as parquet (~35MB, 525K rows at 1-min resolution).

In [ ]:
if run_long_series:
    import pandas as pd, json
    from pathlib import Path

    long_names = ["normal_long", "bearing_failure_long",
                  "impeller_failure_long", "radial_failure_long"]

    # Check if cached data matches current parameters
    config_path = Path("sample_data/long_series_config.json")
    current_config = dict(
        duration_days=duration_days, save_freq_s=save_freq_s,
        decoy_freq_per_day=decoy_freq_per_day, failure_severity=failure_severity,
        maintenance_freq_per_month=maintenance_freq_per_month,
        ambient_var_K=ambient_var_K, duty_cycle_var=duty_cycle_var, seed=seed,
    )

    cache_valid = all(Path(f"sample_data/{n}.parquet").exists() for n in long_names)
    if cache_valid and config_path.exists():
        cached_config = json.loads(config_path.read_text())
        if cached_config != current_config:
            print(f"Parameters changed — regenerating (was {cached_config.get('duration_days')}d, "
                  f"now {duration_days}d)")
            cache_valid = False

    if cache_valid:
        print("Loading existing long series from sample_data/*.parquet")
        long_results = {}
        for n in long_names:
            long_results[n] = pd.read_parquet(f"sample_data/{n}.parquet")
            print(f"  {n}: {len(long_results[n])} rows")
    else:
        print("Generating long series...")
        from sample_generator import generate_dataset
        long_results = generate_dataset(
            duration_days=duration_days,
            save_freq_s=save_freq_s,
            decoy_freq_per_day=decoy_freq_per_day,
            failure_severity=failure_severity,
            maintenance_freq_per_month=maintenance_freq_per_month,
            ambient_var_K=ambient_var_K,
            duty_cycle_var=duty_cycle_var,
            seed=seed,
        )
        config_path.write_text(json.dumps(current_config, indent=2))
else:
    long_results = None
    print("Skipped long series (run_long_series=False)")

In [ ]:
if long_results:
    rows = []
    for name, df in long_results.items():
        vc = df["label"].value_counts()
        n_decoys = int((df["event_type"].str.startswith("decoy_")).sum())
        rows.append({
            "series": name, "rows": len(df),
            "days": f'{len(df)*save_freq_s/86400:.0f}',
            "NORMAL": vc.get("NORMAL", 0),
            "PRE_FAILURE": vc.get("PRE_FAILURE", 0),
            "FAILURE": vc.get("FAILURE", 0),
            "decoy_rows": n_decoys,
        })
    display(pd.DataFrame(rows).set_index("series"))

In [ ]:
def plot_long_series(df, title, signals=None, ds_factor=downsample):
    """Plot a full year-long series with event shading. Downsamples for performance."""
    if signals is None:
        signals = ["shaft_speed_rads", "thrust_bearing_K", "radial_bearing_K",
                    "fluid_temp_K", "flow_out_m3s"]

    ds = df.iloc[::ds_factor]
    time_days = ds["time_h"] / 24.0

    # Color map for event types
    event_colors = {
        "normal": "rgba(0,0,0,0)",
        "decoy_highload_step": "rgba(29,158,117,0.15)",
        "decoy_highload_ramp": "rgba(29,158,117,0.10)",
        "decoy_bp_step": "rgba(33,150,243,0.15)",
        "decoy_bp_ramp": "rgba(33,150,243,0.10)",
    }
    for c in ds["event_type"].unique():
        if c.startswith("failure_"):
            event_colors[c] = "rgba(216,90,48,0.25)"

    fig = make_subplots(
        rows=len(signals), cols=1, shared_xaxes=True,
        subplot_titles=signals, vertical_spacing=0.03,
    )

    for i, sig in enumerate(signals):
        row = i + 1
        fig.add_trace(go.Scattergl(
            x=time_days, y=ds[sig], mode="lines",
            line=dict(width=0.5, color="#333"),
            name=sig, showlegend=False,
        ), row=row, col=1)

        if i == 0:
            for evt in ds["event_type"].unique():
                if evt == "normal":
                    continue
                mask = ds["event_type"] == evt
                if not mask.any():
                    continue
                color = event_colors.get(evt, "rgba(150,150,150,0.15)")
                changes = mask.astype(int).diff().fillna(0)
                starts = time_days[changes == 1].values
                ends = time_days[changes == -1].values
                if mask.iloc[-1]:
                    ends = np.append(ends, time_days.iloc[-1])
                for s, e in zip(starts, ends):
                    fig.add_vrect(x0=s, x1=e, fillcolor=color, layer="below",
                                  line_width=0)

    fig.update_xaxes(title_text="Time (days)", row=len(signals), col=1)
    fig.update_layout(
        title=title,
        height=200 * len(signals),
        template="plotly_white",
        showlegend=False,
    )
    return fig

### Healthy series (365 days, ~525 decoy events, no failure)
Green shading = decoy events. Zoom in with plotly to inspect individual events.

In [ ]:
if long_results:
    fig = plot_long_series(long_results["normal_long"],
                           f"Normal (healthy) \u2014 {duration_days:.0f} days")
    show_fig(fig)

### Bearing failure series (365 days)
Red shading = failure episode (~6h). Green/blue = decoy events throughout.

In [ ]:
if long_results:
    fig = plot_long_series(long_results["bearing_failure_long"],
                           f"Bearing failure \u2014 {duration_days:.0f} days")
    show_fig(fig)

### Impeller failure series (365 days)

In [ ]:
if long_results:
    fig = plot_long_series(long_results["impeller_failure_long"],
                           f"Impeller failure \u2014 {duration_days:.0f} days")
    show_fig(fig)

### Radial bearing failure series (365 days)

In [ ]:
if long_results:
    fig = plot_long_series(long_results["radial_failure_long"],
                           f"Radial bearing failure \u2014 {duration_days:.0f} days")
    show_fig(fig)

### Zoom: failure windows at full resolution
Shows 24h around each failure event — the transition from normal → pre-failure → failure at 1-min resolution.

In [ ]:
def plot_failure_zoom(df, series_name, window_hours=24):
    """Zoom into the failure window at full resolution."""
    fail_idx = df.index[df["label"] == "FAILURE"]
    if len(fail_idx) == 0:
        print(f"  No failure in {series_name}")
        return None
    fail_row = fail_idx[0]
    rows_per_h = 3600 // save_freq_s
    half_window = int(window_hours / 2 * rows_per_h)
    start = max(0, fail_row - half_window)
    end = min(len(df), fail_row + half_window)
    window = df.iloc[start:end]
    signals = ["shaft_speed_rads", "thrust_bearing_K", "radial_bearing_K",
               "fluid_temp_K", "flow_out_m3s"]
    time_h = window["time_h"]
    fig = make_subplots(rows=len(signals), cols=1, shared_xaxes=True,
                        subplot_titles=signals, vertical_spacing=0.04)
    label_colors = {"NORMAL": "#1D9E75", "PRE_FAILURE": "#BA7517", "FAILURE": "#D85A30"}
    for i, sig in enumerate(signals):
        row = i + 1
        for lbl, color in label_colors.items():
            mask = window["label"] == lbl
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=time_h[mask], y=window[sig][mask], mode="markers+lines",
                    marker=dict(size=2, color=color), line=dict(width=1, color=color),
                    name=lbl if i == 0 else None, showlegend=(i == 0), legendgroup=lbl,
                ), row=row, col=1)
        for evt in window["event_type"].unique():
            if evt == "normal":
                continue
            mask = window["event_type"] == evt
            changes = mask.astype(int).diff().fillna(0)
            starts = time_h[changes == 1].values
            ends = time_h[changes == -1].values
            if mask.iloc[-1]:
                ends = np.append(ends, time_h.iloc[-1])
            fc = "rgba(216,90,48,0.12)" if "failure" in evt else "rgba(29,158,117,0.12)"
            for s, e in zip(starts, ends):
                fig.add_vrect(x0=s, x1=e, fillcolor=fc, line_width=0, row=row, col=1)
    fail_day = df.iloc[fail_row]["time_h"] / 24
    fig.update_xaxes(title_text="Time (hours)", row=len(signals), col=1)
    fig.update_layout(
        title=f"{series_name} \u2014 {window_hours}h around failure (day {fail_day:.1f})",
        height=180 * len(signals), template="plotly_white")
    return fig

if long_results:
    for name in ["bearing_failure_long", "impeller_failure_long", "radial_failure_long"]:
        fig = plot_failure_zoom(long_results[name], name)
        if fig:
            show_fig(fig)